# DF40 Dataset — Correction des anomalies D1 (Notebook 02b)

Ce notebook est a utilisé UNE SEULE FOIS à la suite du notebook 02_data_audit.ipynb.


# CELLULE 1 — MONTAGE DRIVE ET CONFIGURATION

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install tqdm pillow pandas -q

import os
import shutil
import random
import pandas as pd
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from datetime import datetime

# ── Seed reproductibilité ─────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ── Chemins ───────────────────────────────────────────────────────────────────
PROJECT_ROOT  = '/content/drive/MyDrive/Memoire_Deepfakes'
DATA_DIR      = f'{PROJECT_ROOT}/data'
RAW_DIR       = f'{DATA_DIR}/raw'
REAL_DIR      = f'{RAW_DIR}/DF40_real'
FAKE_BASE_DIR = f'{RAW_DIR}/DF40_fake_DMs'
MANIFEST_PATH = f'{DATA_DIR}/real_video_manifest.csv'
AUDIT_DIR     = f'{DATA_DIR}/audit'
os.makedirs(AUDIT_DIR, exist_ok=True)

# ── Cibles post-correction ────────────────────────────────────────────────────
TARGET_FAKE_TOTAL = 3_511   # MJ=1595 + ddim=1300 + DiT=358 + SiT=258
TARGET_REAL_TOTAL = 3_511   # Ratio 1:1 maintenu
DM_METHODS_NEW    = ['MidJourney', 'ddim', 'DiT', 'SiT']

# ── Fichiers MidJourney corrompus identifiés en D1 ───────────────────────────
CORRUPT_MJ_FILES = {
    'MidJourney_001073.jpg',
    'MidJourney_001196.jpg',
    'MidJourney_001239.jpg',
    'MidJourney_001251.jpg',
    'MidJourney_001423.jpg',
}

def count_images(directory):
    """Compte les images .jpg/.jpeg/.png dans un dossier."""
    if not os.path.isdir(directory):
        return 0
    return len([f for f in os.listdir(directory)
                if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}])

# ── État initial ──────────────────────────────────────────────────────────────
print("=" * 65)
print("NOTEBOOK 02b — CORRECTION DATASET DF40 v4")
print("=" * 65)
print("\nÉtat initial :")
print(f"  REAL           : {count_images(REAL_DIR):,}")
for m in ['MidJourney', 'ddim', 'DiT', 'SiT', 'CollabDiff']:
    d = os.path.join(FAKE_BASE_DIR, m)
    exist = '✅' if os.path.isdir(d) else '⊘ '
    print(f"  {exist} FAKE/{m:<15s}: {count_images(d):,}")
print("=" * 65)
print("✅ Configuration chargée — 4 étapes à exécuter dans l'ordre")


Mounted at /content/drive
NOTEBOOK 02b — CORRECTION DATASET DF40 v4

État initial :
  REAL           : 3,774
  ✅ FAKE/MidJourney     : 1,600
  ✅ FAKE/ddim           : 1,300
  ✅ FAKE/DiT            : 358
  ✅ FAKE/SiT            : 258
  ✅ FAKE/CollabDiff     : 258
✅ Configuration chargée — 4 étapes à exécuter dans l'ordre


# CELLULE 2 — Center crop MidJourney 1024×1024 → 256×256

In [ ]:
print("=" * 65)
print("ÉTAPE 1 — Center crop MidJourney 1024×1024 → 256×256")
print("=" * 65)

MJ_DIR = os.path.join(FAKE_BASE_DIR, 'MidJourney')
n_cropped = 0
n_already_ok = 0
n_corrupt_deleted = 0
n_other_error = 0

all_mj_files = sorted([
    f for f in os.listdir(MJ_DIR)
    if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}
])

for fname in tqdm(all_mj_files, desc='MidJourney correction'):
    fpath = os.path.join(MJ_DIR, fname)

    # ── Cas 1 : fichier corrompu identifié → suppression directe ─────────────
    if fname in CORRUPT_MJ_FILES:
        os.remove(fpath)
        n_corrupt_deleted += 1
        continue

    try:
        img = Image.open(fpath).convert('RGB')
        w, h = img.size

        # ── Cas 2 : déjà en 256×256 → rien à faire ───────────────────────────
        if (w, h) == (256, 256):
            n_already_ok += 1
            continue

        # ── Cas 3 : ≥ 256×256 → center crop ─────────────────────────────────
        if w >= 256 and h >= 256:
            left   = (w - 256) // 2
            top    = (h - 256) // 2
            right  = left + 256
            bottom = top  + 256
            img_cropped = img.crop((left, top, right, bottom))
            img_cropped.save(fpath, 'JPEG', quality=95)
            n_cropped += 1

        # ── Cas 4 : < 256×256 → irrecuperable, suppression ───────────────────
        else:
            print(f"  ⚠️  {fname} trop petite ({w}×{h}) — supprimée")
            os.remove(fpath)
            n_other_error += 1

    except Exception as e:
        print(f"  ⚠️  Erreur inattendue sur {fname} : {e} — supprimée")
        if os.path.exists(fpath):
            os.remove(fpath)
        n_other_error += 1

mj_final = count_images(MJ_DIR)
print(f"\n  Déjà en 256×256      : {n_already_ok:,}")
print(f"  Center-croppés       : {n_cropped:,}")
print(f"  Corrompus supprimés  : {n_corrupt_deleted}")
print(f"  Autres erreurs       : {n_other_error}")
print(f"  ─────────────────────────────")
flag = '✅' if mj_final == 1_595 else '⚠️'
print(f"  {flag} MidJourney final    : {mj_final:,}  (attendu : 1 595)")


# CELLULE 3 — ÉTAPE 2 : Exclusion de CollabDiff

CollabDiff présentent deux groupes de résolutions incompatibles à nos attentes :
178×218 px (trop petites — upscaling impossible sans introduire d'artefacts artificiels)  et 512×512 px (techniquement croppables, mais source trop petite pour être statistiquement représentative après exclusion du premier groupe).  

In [ ]:
print("=" * 65)
print("ÉTAPE 2 — Exclusion de CollabDiff")
print("=" * 65)

collab_dir = os.path.join(FAKE_BASE_DIR, 'CollabDiff')

if os.path.isdir(collab_dir):
    n_collab = count_images(collab_dir)
    shutil.rmtree(collab_dir)
    print(f"  ✅ Dossier CollabDiff supprimé ({n_collab} fichiers)")
else:
    print("  ℹ️  Dossier CollabDiff déjà absent — étape ignorée")

# Vérification du total FAKE intermédiaire
print("\n  Distribution FAKE intermédiaire :")
total_fake_check = 0
for m in DM_METHODS_NEW:
    cnt = count_images(os.path.join(FAKE_BASE_DIR, m))
    total_fake_check += cnt
    print(f"    {m:<15s}: {cnt:,}")
flag = '✅' if total_fake_check == TARGET_FAKE_TOTAL else '⚠️'
print(f"    {'TOTAL':<15s}: {total_fake_check:,}  (cible : {TARGET_FAKE_TOTAL:,})  {flag}")


# CELLULE 4 — ÉTAPE 3 : Sous-échantillonnage REAL vidéo-level → 3 511

FAKE passe de 3 774 à 3 511, REAL doit être réduit en conséquence afin de conserver le ratio  FAKE / REAL = 1:1.  

In [ ]:
print("=" * 65)
print("ÉTAPE 3 — Sous-échantillonnage REAL vidéo-level → 3 511")
print("=" * 65)

manifest = pd.read_csv(MANIFEST_PATH)
print(f"  Manifeste chargé : {len(manifest)} lignes")
print(f"  Colonnes         : {list(manifest.columns)}")

# Clé unique = prefix + video_id (entier)
manifest['unique_vid'] = manifest['prefix'].astype(str) + '_' + manifest['video_id'].astype(str)
all_unique_vids = manifest['unique_vid'].unique().tolist()
print(f"  Vidéos uniques   : {len(all_unique_vids):,}")

n_real_current   = count_images(REAL_DIR)
n_to_remove      = n_real_current - TARGET_REAL_TOTAL
frames_per_vid   = 2
# Division plafond : garantit qu'on n'excède pas la cible
n_vids_to_remove = (n_to_remove + frames_per_vid - 1) // frames_per_vid

print(f"\n  REAL actuels     : {n_real_current:,}")
print(f"  REAL cible       : {TARGET_REAL_TOTAL:,}")
print(f"  Images à retirer : ~{n_to_remove}")
print(f"  Vidéos à retirer : {n_vids_to_remove}")

if n_to_remove <= 0:
    print("\n  ℹ️  REAL déjà à la cible ou en dessous — étape ignorée")
else:
    # Sélection aléatoire reproductible
    random.seed(RANDOM_SEED)
    random.shuffle(all_unique_vids)
    vids_to_remove = set(all_unique_vids[:n_vids_to_remove])

    # Liste des fichiers à supprimer depuis le manifeste
    files_to_remove = manifest[
        manifest['unique_vid'].isin(vids_to_remove)
    ]['filename'].tolist()
    print(f"  Fichiers ciblés  : {len(files_to_remove)}")

    # Suppression physique
    n_deleted   = 0
    n_not_found = 0
    for fname in tqdm(files_to_remove, desc='Suppression REAL'):
        fpath = os.path.join(REAL_DIR, fname)
        if os.path.exists(fpath):
            os.remove(fpath)
            n_deleted += 1
        else:
            n_not_found += 1

    print(f"  ✅ Supprimés     : {n_deleted}")
    if n_not_found > 0:
        print(f"  ⚠️  Introuvables : {n_not_found} — vérifier les noms dans le manifeste")

    # Mise à jour et sauvegarde du manifeste
    manifest_updated = manifest[
        ~manifest['unique_vid'].isin(vids_to_remove)
    ].copy().drop(columns=['unique_vid'])
    manifest_updated.to_csv(MANIFEST_PATH, index=False)
    print(f"  ✅ Manifeste mis à jour : {len(manifest_updated)} lignes")

# Vérification
real_final = count_images(REAL_DIR)
flag = '✅' if real_final == TARGET_REAL_TOTAL else '⚠️'
print(f"\n  {flag} REAL final : {real_final:,}  (cible : {TARGET_REAL_TOTAL:,})")
if real_final != TARGET_REAL_TOTAL:
    print(f"     Écart de {real_final - TARGET_REAL_TOTAL:+d} images")
    print("     Cause possible : frames_per_video ≠ 2 pour certaines vidéos")
    print("     → Ajuster n_vids_to_remove manuellement si nécessaire")


# CELLULE 4b — Ajustement FAKE : suppression de 1 image MidJourney
Raison : 263 images REAL à retirer est impair → impossible à respecter
exactement avec un sous-échantillonnage vidéo-level à 2 frames/vidéo.

REAL final = 3 510. On ajuste FAKE de 3 511 → 3 510 pour maintenir ratio 1:1.

On supprime la dernière image MidJourney (indice le plus élevé, seed-neutre).

In [ ]:


import os
from pathlib import Path

MJ_DIR = '/content/drive/MyDrive/Memoire_Deepfakes/data/raw/DF40_fake_DMs/MidJourney'

mj_files = sorted([
    f for f in os.listdir(MJ_DIR)
    if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}
])

file_to_remove = mj_files[-1]   # dernière image MidJourney (index le plus élevé)
os.remove(os.path.join(MJ_DIR, file_to_remove))

print(f"  ✅ Supprimé : {file_to_remove}")

# Vérification finale
real_final = len([f for f in os.listdir(
    '/content/drive/MyDrive/Memoire_Deepfakes/data/raw/DF40_real')
    if Path(f).suffix.lower() in {'.jpg','.jpeg','.png'}])
fake_final = sum(
    len([f for f in os.listdir(
        f'/content/drive/MyDrive/Memoire_Deepfakes/data/raw/DF40_fake_DMs/{m}')
         if Path(f).suffix.lower() in {'.jpg','.jpeg','.png'}])
    for m in ['MidJourney','ddim','DiT','SiT']
)
ratio = real_final / fake_final
print(f"  REAL  : {real_final:,}")
print(f"  FAKE  : {fake_final:,}")
print(f"  TOTAL : {real_final + fake_final:,}")
print(f"  Ratio : {ratio:.4f}  {'✅' if abs(ratio-1.0) < 0.001 else '⚠️'}")

# CELLULE 5 — VÉRIFICATION FINALE ET RAPPORT DE CORRECTION

In [ ]:
print("=" * 65)
print("VÉRIFICATION FINALE")
print("=" * 65)

real_final  = count_images(REAL_DIR)
fake_counts = {m: count_images(os.path.join(FAKE_BASE_DIR, m)) for m in DM_METHODS_NEW}
fake_final  = sum(fake_counts.values())
grand_total = real_final + fake_final
ratio       = real_final / max(fake_final, 1)

manifest_final = pd.read_csv(MANIFEST_PATH)

print(f"\n  REAL   : {real_final:,}  {'✅' if real_final == TARGET_REAL_TOTAL else '⚠️  ÉCART : ' + str(real_final - TARGET_REAL_TOTAL)}")
print(f"  FAKE   : {fake_final:,}  {'✅' if fake_final == TARGET_FAKE_TOTAL else '⚠️  ÉCART : ' + str(fake_final - TARGET_FAKE_TOTAL)}")
print(f"  TOTAL  : {grand_total:,}")
print(f"  Ratio  : {ratio:.4f}  {'✅' if abs(ratio - 1.0) < 0.01 else '⚠️'}")
print(f"  Manifeste : {len(manifest_final)} lignes")

print("\n  Distribution FAKE :")
DM_EXPECTED_NEW = {'MidJourney': 1_595, 'ddim': 1_300, 'DiT': 358, 'SiT': 258}
for m in DM_METHODS_NEW:
    cnt = fake_counts[m]
    exp = DM_EXPECTED_NEW[m]
    flag = '✅' if cnt == exp else '⚠️'
    print(f"    {flag} {m:<15s}: {cnt:,}  (attendu : {exp:,})")

# ── Génération du rapport ─────────────────────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
report_path = f'{AUDIT_DIR}/correction_report_{ts}.txt'

overall_ok = (
    real_final == TARGET_REAL_TOTAL and
    fake_final == TARGET_FAKE_TOTAL and
    abs(ratio - 1.0) < 0.01
)

with open(report_path, 'w', encoding='utf-8') as f:
    sep = '=' * 65
    f.write(sep + '\n')
    f.write("RAPPORT DE CORRECTION — DF40 v4\n")
    f.write("Mémoire : Obsolescence des détecteurs de deepfakes\n")
    f.write("Auteur  : Maxime Ducarme\n")
    f.write(sep + '\n')
    f.write(f"Date    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Seed    : {RANDOM_SEED}\n\n")

    f.write("CORRECTIONS APPLIQUÉES\n")
    f.write('-' * 65 + '\n')
    f.write("  1. MidJourney — Center crop 1024×1024 → 256×256\n")
    f.write("     Indices 981–1599 (614 images). Standard DeepfakeBench.\n")
    f.write("  2. MidJourney — Suppression 5 fichiers corrompus\n")
    f.write("     001073, 001196, 001239, 001251, 001423\n")
    f.write("  3. CollabDiff — Exclusion totale (résolutions hétérogènes)\n")
    f.write("     178×218 px (125 img) + 512×512 px (133 img) = 258 img\n")
    f.write("  4. REAL — Sous-échantillonnage vidéo-level (seed=42)\n")
    f.write("     3 774 → 3 511 images. Ratio 1:1 maintenu.\n\n")

    f.write("RÉSULTAT FINAL\n")
    f.write('-' * 65 + '\n')
    f.write(f"  REAL    : {real_final:,}  (cible : {TARGET_REAL_TOTAL:,})\n")
    for m in DM_METHODS_NEW:
        f.write(f"  FAKE/{m:<13s}: {fake_counts[m]:,}  (cible : {DM_EXPECTED_NEW[m]:,})\n")
    f.write(f"  TOTAL   : {grand_total:,}\n")
    f.write(f"  Ratio   : {ratio:.4f}\n")
    f.write(f"  Manifeste : {len(manifest_final)} lignes\n\n")

    f.write(f"  VERDICT : {'✅ DATASET CORRIGÉ — Relancer D1/D2 du notebook 02 pour certification' if overall_ok else '⚠️  ÉCARTS DÉTECTÉS — Vérifier les étapes ci-dessus'}\n")
    f.write(sep + '\n')

print(f"\n  ✅ Rapport sauvegardé : {report_path}")
print()
if overall_ok:
    print("  ✅ CORRECTION TERMINÉE AVEC SUCCÈS")
    print("  Prochaines étapes :")
    print("    [1] Relancer les cellules D1 et D2 du notebook 02 (validation)")
    print("    [2] Créer et exécuter le notebook 03 (split vidéo-level 20/40/40)")
else:
    print("  ⚠️  Des écarts subsistent — consulter le rapport et corriger")
print("=" * 65)
